In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
proportions_file = 'TOO-Decon-Noise/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon-Noise/'
results_dir = './Results-Pearson-PredictedVSActual'
os.makedirs(results_dir, exist_ok=True)

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Add missing columns and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0
prop_df_tissues = prop_df[reference_tissues].div(prop_df[reference_tissues].sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*_500', '*_modified.txt')))

results = []  # collect summary rows

for file_path in decon_files:
    try:
        # === Extract info from directory name ===
        dir_name = os.path.basename(os.path.dirname(file_path))
        dir_parts = dir_name.replace("Decon-Results_", "").split("_")

        # Extract NoiseLevel
        noise_level = 'NA'
        for part in dir_parts:
            if part.startswith("Noise"):
                noise_level = part.replace("Noise", "")
                break

        # Extract reference matrix (last 3 parts)
        matrix = '_'.join(dir_parts[-3:]) if len(dir_parts) >= 3 else dir_name

        # Get method from filename
        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        # === Load and align data ===
        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)
        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0
        decon_df = decon_df[reference_tissues]

        common_samples = prop_df.index.intersection(decon_df.index)
        true_vals_df = prop_df.loc[common_samples, reference_tissues]
        pred_vals_df = decon_df.loc[common_samples, reference_tissues]

        # Flatten and filter
        true_vals_flat = true_vals_df.values.flatten()
        pred_vals_flat = pred_vals_df.values.flatten()
        mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
        true_vals_flat = true_vals_flat[mask_nonzero]
        pred_vals_flat = pred_vals_flat[mask_nonzero]

        # Pearson correlation
        r, p = pearsonr(true_vals_flat, pred_vals_flat)
        r2 = r**2
        print(f"{method}, Noise {noise_level}, Matrix {matrix} — Pearson r = {r:.4f}, p = {p:.2e}")

        # Save result row
        results.append([matrix, method, noise_level, r, r2, p])

        # === Plot ===
        plot_name = f'{method}_Noise{noise_level}.png'
        plot_path = os.path.join(results_dir, plot_name)

        plt.figure(figsize=(8, 8))
        sns.set(style="white", context="talk")
        ax = sns.regplot(
            x=true_vals_flat,
            y=pred_vals_flat,
            scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
            line_kws={'color': 'crimson', 'lw': 2}
        )
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Actual Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Predicted Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_title(f'{method} — Noise {noise_level}\nPearson r = {r:.2f}, p = {p:.2e}',
                     fontsize=16, fontweight='bold')
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        sns.despine()
        plt.tight_layout()
        plt.savefig(plot_path, dpi=300)
        plt.close()
        print(f"Saved plot: {plot_path}")
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# === SAVE SUMMARY TABLE (append if exists) ===
if results:
    results_df = pd.DataFrame(
        results,
        columns=["Matrix", "Method", "NoiseLevel", "Pearson_r", "R_squared", "p_value"]
    )
    results_csv = os.path.join(results_dir, "Random_500_Pearson_Summary.csv")

    if os.path.exists(results_csv):
        old_df = pd.read_csv(results_csv)
        combined_df = pd.concat([old_df, results_df], ignore_index=True)
        combined_df.to_csv(results_csv, index=False)
        print(f"Appended Pearson summary table: {results_csv}")
    else:
        results_df.to_csv(results_csv, index=False)
        print(f"Created Pearson summary table: {results_csv}")
else:
    print("No results to save.")


BayesPrism, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.6100, p = 0.00e+00
Saved plot: ./Results-Pearson-PredictedVSActual/BayesPrism_Noise0.1.png
MuSiC, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.1766, p = 8.45e-122
Saved plot: ./Results-Pearson-PredictedVSActual/MuSiC_Noise0.1.png
ReDeconv, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.6312, p = 0.00e+00
Saved plot: ./Results-Pearson-PredictedVSActual/ReDeconv_Noise0.1.png
CIBERSORTx, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.4110, p = 0.00e+00
Saved plot: ./Results-Pearson-PredictedVSActual/CIBERSORTx_Noise0.1.png
NNLS, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.2187, p = 1.70e-127
Saved plot: ./Results-Pearson-PredictedVSActual/NNLS_Noise0.1.png
QP, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.2708, p = 1.04e-245
Saved plot: ./Results-Pearson-PredictedVSActual/QP_Noise0.1.png
nuSVR, Noise 0.1, Matrix 2Median_300_500 — Pearson r = 0.5665, p = 0.00e+00
Saved plot: ./Results-Pearson-PredictedVSActu

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'Results-Pearson-PredictedVSActual/Random_500_Pearson_Summary.csv'
df = pd.read_csv(file_path)

# === PIVOT TO METHOD × NOISE LEVEL ===
pivot_df = df.pivot_table(
    index='Method',
    columns='NoiseLevel',
    values='Pearson_r',    
    aggfunc='mean'
).sort_index(axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='coolwarm',             
#    annot=True,
#    fmt=".1f",
#    annot_kws={"size": 10, "weight": "normal"},  # consistent with MAE heatmap
    cbar_kws={'label': 'Pearson r', 'shrink': 0.9},
    linewidths=0,
    linecolor='white',
    center=0
)

# === AXES STYLING ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Noise Level', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === COLORBAR STYLING ===
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Pearson r", fontsize=13, rotation=270, labelpad=20)
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === TICK STYLING ===
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# === SAVE FIGURE ===
fig.savefig("Pearson_Heatmap_Best-Matrix_Noise_500_Clean.svg")
plt.close(fig)